# Single-Stage Axial Compressor — Meanline Design Analysis

**Configuration:** IGV + Rotor + Stator, single stage, rotating rig  
**Method:** Meanline (through-flow) analysis at the mean streamline  
**Scope:** Parametric sweep → design point selection → velocity triangles → radial distribution

---

All physics and plotting logic lives in `src/`. This notebook contains only:
- design parameters you edit
- function calls
- results and figures

| Module | Contents |
|---|---|
| `src/constants.py` | Air properties, ISA conditions |
| `src/meanline.py` | `meanline_analysis()` — core Euler calculation |
| `src/radial.py` | `free_vortex()` — radial distribution |
| `src/plotting.py` | All figures |

---
## Imports

Run this first. `sys.path.insert` lets Python find the `src/` package one level above the `notebooks/` folder.

In [1]:
import sys
sys.path.insert(0, '../')

import pandas as pd
from itertools import product

from src import (
    meanline_analysis,
    free_vortex,
    plot_design_maps,
    plot_velocity_triangles,
    plot_radial,
)

print('Imports OK')

ImportError: cannot import name 'meanline_analysis' from 'src' (c:\dev\rotating-rig-design\notebooks\..\src\__init__.py)

---
## Design Parameters

> **Start here.** Edit any value in this cell to explore a different design space.
> All subsequent cells pick up these variables automatically.

### Geometric sweep
| Parameter | Symbol | Range | Unit |
|---|---|---|---|
| Tip diameter | D_tip | 700 – 1000 | mm |
| Rotational speed | N | 3000 – 4000 | RPM |
| Pressure ratio | PR | 1.10 – 1.20 | — |

### Fixed design assumptions
| Parameter | Symbol | Value | Rationale |
|---|---|---|---|
| Min. blade height | h_min | 50 mm | Mechanical and aero viability |
| Hub-to-tip ratio | ν | 0.70 | Typical for low-PR single stage |
| Isentropic efficiency | η_is | 0.87 | Realistic target for this class |

In [ ]:
# --- Sweep ranges ---
D_tip_range = [0.70, 0.80, 0.90, 1.00]   # tip diameters [m]
RPM_range   = [3000, 3500, 4000]           # rotational speeds [RPM]
PR_range    = [1.10, 1.15, 1.20]           # pressure ratios [-]

# --- Fixed assumptions ---
h_min     = 0.05   # minimum blade height [m]
nu_target = 0.70   # hub-to-tip ratio r_hub / r_tip [-]
eta_is    = 0.87   # isentropic efficiency [-]

print(f'Tip diameters         : {[f"{d*1000:.0f} mm" for d in D_tip_range]}')
print(f'RPM options           : {RPM_range}')
print(f'PR targets            : {PR_range}')
print(f'Total combinations    : {len(D_tip_range) * len(RPM_range) * len(PR_range)}')

---
## Parametric Sweep

Evaluates every combination of `D_tip × RPM × PR` using the Euler turbomachinery equation.
Results are collected into a single DataFrame for filtering and plotting.

Each row contains: annulus geometry, blade velocities, dimensionless coefficients
(ψ, φ, De Haller), mass flow rate, shaft power, and blade angles at the mean radius.

In [ ]:
results = [
    meanline_analysis(D_tip, N, PR, eta_is, nu=nu_target, h_min=h_min)
    for D_tip, N, PR in product(D_tip_range, RPM_range, PR_range)
]

df = pd.DataFrame(results)

print('Sweep complete.')
print(f'  Total combinations  : {len(df)}')
print(f'  h >= 50 mm          : {df["h_ok"].sum()}')
print(f'  De Haller >= 0.72   : {(df["De_Haller"] >= 0.72).sum()}')
print(f'  Power range         : {df["P_shaft_kW"].min():.1f} – {df["P_shaft_kW"].max():.1f} kW')

---
## Design Maps

Four panels, read left-to-right, top-to-bottom:

1. **Shaft power** — determines which motor class is needed
2. **Smith chart** — shows whether design points fall in the high-efficiency zone
3. **Blade height** — confirms the 50 mm floor is met
4. **De Haller number** — flags combinations at risk of diffusion stall

Line style encodes pressure ratio: solid = PR 1.15, dashed = PR 1.10 and 1.20.  
Colour encodes RPM: green = 3000, blue = 3500, orange = 4000.

In [ ]:
plot_design_maps(df, eta_is, nu_target, h_min, save_path='design_maps.png')

---
## Design Point Selection

Filter the full sweep down to viable candidates using the criteria below.
Results are sorted by shaft power (lowest first) — the preferred candidate is `iloc[0]`.

> Widen any range or raise the power budget if no candidates appear.

In [ ]:
# --- Selection criteria (edit these) ---
CRITERIA = {
    'h_ok':       True,          # blade height >= h_min
    'De_Haller':  (0.72, 1.00),  # stall margin  [De Haller 1953]
    'phi':        (0.35, 0.65),  # flow coefficient — Smith chart optimum
    'psi':        (0.15, 0.55),  # work coefficient — Smith chart optimum
    'M_tip':      (0.00, 0.60),  # tip Mach — subsonic only
    'P_shaft_kW': (0.00, 200.0), # motor power budget [kW]
}

# --- Apply filter ---
mask = (
    (df['h_ok']       == CRITERIA['h_ok']) &
    (df['De_Haller']  .between(*CRITERIA['De_Haller'])) &
    (df['phi']        .between(*CRITERIA['phi'])) &
    (df['psi']        .between(*CRITERIA['psi'])) &
    (df['M_tip']      .between(*CRITERIA['M_tip'])) &
    (df['P_shaft_kW'] .between(*CRITERIA['P_shaft_kW']))
)

df_sel = df[mask].sort_values('P_shaft_kW').reset_index(drop=True)

print(f'{len(df_sel)} candidates satisfy all criteria.')
print()

display_cols = [
    'D_tip_mm', 'N_RPM', 'PR', 'h_mm',
    'phi', 'psi', 'De_Haller',
    'mdot_kg_s', 'P_shaft_kW',
    'beta1_deg', 'beta2_deg', 'alpha2_deg',
]
df_sel[display_cols].round(2)

---
## Preferred Design Point Summary

Prints a full summary of the lowest-power candidate (`iloc[0]` from the selection above).  
Motor rated power includes a **+20% safety margin** for catalogue selection.

In [ ]:
SAFETY_MARGIN = 1.20

best = df_sel.iloc[0]

print('Preferred design point')
print('=' * 40)
print(f'  D_tip       = {best["D_tip_mm"]:.0f} mm')
print(f'  r_mean      = {best["r_mean"]*1000:.1f} mm')
print(f'  h           = {best["h_mm"]:.1f} mm')
print(f'  N           = {best["N_RPM"]:.0f} RPM')
print(f'  PR          = {best["PR"]:.2f}')
print()
print(f'  U_mean      = {best["U_mean"]:.1f} m/s')
print(f'  phi (φ)     = {best["phi"]:.3f}')
print(f'  psi (ψ)     = {best["psi"]:.3f}')
print(f'  De Haller   = {best["De_Haller"]:.3f}')
print(f'  M_tip       = {best["M_tip"]:.3f}')
print()
print(f'  mass flow   = {best["mdot_kg_s"]:.2f} kg/s')
print(f'  P_shaft     = {best["P_shaft_kW"]:.1f} kW')
print(f'  P_motor     >= {best["P_shaft_kW"] * SAFETY_MARGIN:.0f} kW  (+{(SAFETY_MARGIN-1)*100:.0f}% margin)')
print()
print(f'  beta1 (β₁)  = {best["beta1_deg"]:.1f}°   rotor inlet relative angle')
print(f'  beta2 (β₂)  = {best["beta2_deg"]:.1f}°   rotor exit relative angle')
print(f'  alpha2 (α₂) = {best["alpha2_deg"]:.1f}°   stator inlet absolute angle')
print(f'  Delta_beta  = {best["delta_beta_deg"]:.1f}°   rotor deflection')

---
## Velocity Triangles

Shows the rotor inlet (Station 1) and exit (Station 2) velocity diagrams at the mean radius.

- **Red arrow — U** peripheral (blade) speed
- **Blue arrow — C** absolute velocity
- **Green arrow — W** relative velocity, where W = C − U

Inlet assumes swirl-free flow (C_θ1 = 0), consistent with the IGV at neutral setting.

In [ ]:
# Re-run meanline at the selected point to get the full result dictionary
res = meanline_analysis(
    D_tip  = best['D_tip_mm'] / 1000,
    N      = best['N_RPM'],
    PR     = best['PR'],
    eta_is = eta_is,
    nu     = nu_target,
)

plot_velocity_triangles(res, save_path='velocity_triangles.png')

---
## Radial Distribution (Free Vortex)

Applies the free vortex law — r · C_θ = constant — to distribute the meanline
solution from hub to tip across 50 radial stations.

Three panels:
1. **Blade angles** β₁, β₂, α₂ vs span — direct input to blade profile design
2. **Deflection Δβ and De Haller** — checks stall margin holds across the full span
3. **Lieblein diffusion factor DF** — should stay below 0.45 everywhere

> DF uses assumed solidity σ = 1.0. Update once blade count and chord are decided.

In [ ]:
rad = free_vortex(res, n_stations=50, sigma=1.0)

plot_radial(res, rad, save_path='radial_distribution.png')

---
## Motor Selection Summary

Groups all valid combinations by motor power class.  
Only combinations satisfying h >= h_min **and** De Haller >= 0.72 are shown.

In [ ]:
SAFETY_MARGIN = 1.20   # +20% for motor catalogue sizing

MOTOR_CLASSES = [
    (0,    45,   '<= 45 kW',   'Standard industrial'),
    (45,   75,   '45 – 75 kW', 'Medium power'),
    (75,   110,  '75 – 110 kW','High power'),
    (110,  160,  '110 – 160 kW','Large industrial'),
    (160,  9999, '> 160 kW',   'Heavy duty / custom'),
]

df_valid = df[
    df['h_ok'] & (df['De_Haller'] >= 0.72)
].copy()

df_valid['P_motor_kW'] = (df_valid['P_shaft_kW'] * SAFETY_MARGIN).round(0)

print(f'Valid combinations (h OK and DH >= 0.72): {len(df_valid)}')
print()

cols = ['D_tip_mm','N_RPM','PR','h_mm','phi','psi',
        'De_Haller','mdot_kg_s','P_shaft_kW','P_motor_kW']

for lo, hi, label, desc in MOTOR_CLASSES:
    sub = df_valid[
        (df_valid['P_shaft_kW'] > lo) &
        (df_valid['P_shaft_kW'] <= hi)
    ].sort_values('P_shaft_kW')

    if sub.empty:
        continue

    print(f'{label}  —  {desc}  ({len(sub)} combinations)')
    print(sub[cols].round(2).to_string(index=False))
    print()

---
## Next Steps

Once a design point is confirmed, the recommended progression is:

1. **Solidity selection** — use the Lieblein diffusion factor with actual blade count to update DF
2. **Blade profile** — NACA 65-series or controlled-diffusion airfoil (CDA); set camber from β₁, β₂ and Carter's deviation rule
3. **3D CAD model** — parametric blade stacked on the free vortex radial distribution
4. **Instrumentation layout** — pressure taps on stator blade + 5-hole probe traverse plan

---
*v1.2 — Refactored. Physics in `src/`, parameters and results here.*